In [1]:
!pip install torch transformers pytorch pytorch-pretrained-bert

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     |████████████████████████████████| 4.4 MB 25.4 MB/s 
     |████████████████████████████████| 123 kB 65.4 MB/s 
     |████████████████████████████████| 6.6 MB 65.2 MB/s 
     |████████████████████████████████| 596 kB 64.2 MB/s 
     |████████████████████████████████| 101 kB 9.2 MB/s 
     |████████████████████████████████| 132 kB 74.3 MB/s 
     |████████████████████████████████| 79 kB 8.5 MB/s 
     |████████████████████████████████| 9.0 MB 46.1 MB/s 
     |████████████████████████████████| 139 kB 76.8 MB/s 
     |████████████████████████████████| 127 kB 74.7 MB/s 
  ERROR: Failed building wheel for pytorch
  Running setup.py clean for pytorch
Failed to build pytorch
  Attempting uninstall: urllib3
    Found existing installation: urllib3 1.24.3
    Uninstalling urllib3-1.24.3:
      Successfully uninstalled urllib3-1.24.3
  Attempting uninstall: pyyaml
    Found existing installatio

In [2]:
import pandas as pd
import numpy as np
import os
import random
import torch
import transformers
from transformers import BertTokenizer,BertForTokenClassification
#import utils

In [3]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

Loading BERT tokenizer...


Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/570 [00:00<?, ?B/s]

Importing Data

In [8]:
df = pd.read_csv('/content/sentences.txt',header = None , sep  ='., /',encoding = 'utf-8')
df.head()

/usr/local/lib/python3.7/dist-packages/pandas/util/_decorators.py:311: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return func(*args, **kwargs)


,0
0,"complex lange ##vin ( cl ) [ 1 , 2 ] sign prob..."
1,nuclear theory thermal ##ization nuclear react...
2,dirac equation . the cre ##utz model [ 32 ] tr...
3,statistical mechanics nonlinear pde ##s theory...
4,"fluctuating vacuum quantum fields , free maxwe..."


Initial Analysis

In [9]:
df.columns = df.columns.astype('string') # Converting column  data type from object to string
df.columns = df.columns.str.replace('0', 'value') # to change the name of the column for better understanding
df['value'] = df['value'].astype('string')

In [10]:
df.head()

,value
0,"complex lange ##vin ( cl ) [ 1 , 2 ] sign prob..."
1,nuclear theory thermal ##ization nuclear react...
2,dirac equation . the cre ##utz model [ 32 ] tr...
3,statistical mechanics nonlinear pde ##s theory...
4,"fluctuating vacuum quantum fields , free maxwe..."


In [11]:
df.shape

(398, 1)

In [12]:
df.dtypes

value    string
dtype: object

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   value   398 non-null    string
dtypes: string(1)
memory usage: 3.2 KB


Tokenization

In [14]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

Loading BERT tokenizer...


In [20]:
# Print the original sentence.
print(' Original: ', df.value[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(df.value[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(df.value[0])))

 Original:  complex lange ##vin ( cl ) [ 1 , 2 ] sign problem numerical simulations of lattice field theories weight , sampling . nonzero chemical potential , lower and four - dimensional field theories sign problem in the thermodynamic limit [ 3 – 8 ] ( for reviews , e . g . refs . [ 9 , 10 ] ) . however , inc ##epti ##on , [ 11 – 16 ] . improved understanding , relying on the combination of analytical and numerical insight . past , probability distribution complex ##ified configuration space , lange ##vin process , [ 17 , 18 ] . distribution local ##ised cl results . importantly , non ##abel ##ian gauge theories , sl ( n , c ) gauge cooling [ 8 , 10 ] .
Tokenized:  ['complex', 'lange', '#', '#', 'vin', '(', 'cl', ')', '[', '1', ',', '2', ']', 'sign', 'problem', 'numerical', 'simulations', 'of', 'lattice', 'field', 'theories', 'weight', ',', 'sampling', '.', 'non', '##zer', '##o', 'chemical', 'potential', ',', 'lower', 'and', 'four', '-', 'dimensional', 'field', 'theories', 'sign', 'p

In [21]:
max_len = 0

# For every sentence...
for val in df.value:

    # Tokenize the text and add `[CLS]` and `[SEP]` tokens.
    input_ids = tokenizer.encode(val, add_special_tokens=True)

    # Update the maximum sentence length.
    max_len = max(max_len, len(input_ids))

print('Max sentence length: ', max_len)

Max sentence length:  487


In [22]:
# Tokenize all of the sentences and map the tokens to thier word IDs.
input_ids = []
attention_masks = []

# For every sentence...
for val in df.value:
    # `encode_plus` will:
    #   (1) Tokenize the sentence.
    #   (2) Prepend the `[CLS]` token to the start.
    #   (3) Append the `[SEP]` token to the end.
    #   (4) Map tokens to their IDs.
    #   (5) Pad or truncate the sentence to `max_length`
    #   (6) Create attention masks for [PAD] tokens.
    encoded_dict = tokenizer.encode_plus(
                        val,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = 64,           # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )
    
    # Add the encoded sentence to the list.    
    input_ids.append(encoded_dict['input_ids'])
    
    # And its attention mask (simply differentiates padding from non-padding).
    attention_masks.append(encoded_dict['attention_mask'])

# Convert the lists into tensors.
input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)

# Print sentence 0, now as a list of IDs.
print('Original: ', len(df.value))
print('Token IDs:', input_ids[0])

/usr/local/lib/python3.7/dist-packages/transformers/tokenization_utils_base.py:2307: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  FutureWarning,


Original:  398
Token IDs: tensor([  101,  3375, 21395,  1001,  1001, 19354,  1006, 18856,  1007,  1031,
         1015,  1010,  1016,  1033,  3696,  3291, 15973, 24710,  1997, 17779,
         2492,  8106,  3635,  1010, 16227,  1012,  2512,  6290,  2080,  5072,
         4022,  1010,  2896,  1998,  2176,  1011,  8789,  2492,  8106,  3696,
         3291,  1999,  1996,  1996, 10867,  7716, 18279,  7712,  5787,  1031,
         1017,  1516,  1022,  1033,  1006,  2005,  4391,  1010,  1041,  1012,
         1043,  1012, 25416,   102])


In [23]:
from transformers import BertForSequenceClassification

In [24]:
model = BertForSequenceClassification.from_pretrained("textattack/bert-base-uncased-yelp-polarity")


Downloading:   0%|          | 0.00/520 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/418M [00:00<?, ?B/s]

In [29]:
inputs = tokenizer(val,return_tensors="pt")
inputs

{'input_ids': tensor([[  101,  1996, 10867,  2080,  1011, 19577,  2009,  2121,  6993,  1010,
          6258,  3998,  1516,  4278,  1080,  1001,  1001,  1039, 22764, 11520,
          2291,  1031,  2539,  1010,  2322,  1033,  1010, 28667,  2239,  1001,
          1001,  4487,  3508,  1001,  1001, 13749, 12123,  3169,  7722,  1031,
          2184,  1033,  1012,  2174,  1010,  6258,  1010, 11520, 10868,  1010,
          2640,  1012,  3949,  1997,  2009,  2121,  6351,  2522,  1011, 10042,
          1031,  1015,  1516,  1017,  1033,  1010,  1080,  1001,  1001,  1039,
         17856,  1001,  1001, 17153,  3263,  1516, 17528,  1080,  1001,  1001,
          1039,  2364,  2813,  6556,  3033,  1012,  2556,  1010,  2717,  2099,
          1001,  1001,  7110,  2015, 13012,  1001,  1001, 14841,  2819,  2522,
          1011, 14140,  6351,  9014,  1010,  2009,  2121,  6351,  4475, 17856,
          1001,  1001, 17153, 15665,  1012,  6600,  1010,  4517, 10077,  5733,
          1006,  4487,  1001,  1001,  

In [30]:
with torch.no_grad():
    logits = model(**inputs).logits


In [31]:
predicted_class_id = logits.argmax().item()
model.config.id2label[predicted_class_id]

'LABEL_0'